# Homework 2 (Part 2): Markov Decision Processes on FrozenLake

**ESE 561, Theory of Artificial Intelligence.** Total: **10 points**. Submit the executed notebook.

In class we solved the 4×3 world with **value iteration** and **policy iteration**. Here you do the same for FrozenLake, a grid of ice where the agent must reach the goal `G` without falling into holes `H`:

```
SFFF      S = start, F = frozen (safe), H = hole (terminal, reward 0), G = goal (terminal, reward 1)
FHFH      actions: 0 = Left, 1 = Down, 2 = Right, 3 = Up
FFFH      slippery ice: the agent moves in the intended direction with probability 1/3,
HFFG      and slides to each of the two perpendicular directions with probability 1/3
```

Everything in Parts 1–5 uses the **known model** $P(s'\mid s,a)$ and $R(s,a,s')$: this is planning, not learning. Part 6 checks the plans by actually playing the game.

| Part | Topic | Points |
|---|---|---|
| 0 | Understanding environment | 0.2 |
| 1 | The model as arrays; Q-values and the Bellman update | 0.8 |
| 2 | Policy evaluation: exact and by simple iterations | 1.5 |
| 3 | Policy iteration | 0 |
| 4 | Value iteration | 2 |
| 5 | The discount factor changes the policy | 1 |
| 6 | Checking by simulation | 1 |
| 7 | The contraction bound in practice | 1 |
| bonus | | 2.5 |

In [1]:
# !pip install "gymnasium[toy-text]"      # uncomment if Gymnasium is not installed
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")  # harmless pygame warning
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import gymnasium as gym
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 0. The environment (provided)

We use Gymnasium's `FrozenLake-v1` (`pip install "gymnasium[toy-text]"`). Two built-in maps are available, `"4x4"` and `"8x8"`; the time limit is 100 steps on 4×4 and 200 on 8×8. `env.unwrapped.P` is the transition table (the known model), and `env.unwrapped.desc` is the map, and `env.render()` draws it.

In [9]:
# actions: 0 = Left, 1 = Down, 2 = Right, 3 = Up;  render_mode="rgb_array" makes env.render() return the picture
def make_env(map_name="4x4", is_slippery=True):
    return gym.make("FrozenLake-v1", map_name=map_name, is_slippery=is_slippery, render_mode="rgb_array")


env = make_env("4x4", is_slippery=True)
print(env.unwrapped.desc.astype(str))
print('transitions from state 0 (start) under action Right:', env.unwrapped.P[0][2])

## Q0.1 (0.1 pt) Print transitions from state 0 under action "Left"

## Q0.2 (0.1 pt) Print transitions from state 5 under action "Right"

`env.unwrapped.P[s][a]` is a list of tuples `(probability, next_state, reward, terminated)`. States are numbered row by row: state $s$ is at row `s // n_cols`, column `s % n_cols`, with row 0 at the **top**.

The helper below turns this table into dense arrays, the same format as the 4×3 notebook:
`P[s, a, s'] = P(s'|s,a)`, `R[s, a, s'] = R(s,a,s')`, and a boolean vector `terminal`.

In [8]:
def get_model(env):
    '''Dense arrays P[s,a,s'], R[s,a,s'] and the terminal-state mask from env.unwrapped.P.'''
    table = env.unwrapped.P
    nS, nA = len(table), len(table[0])
    P, R = np.zeros((nS, nA, nS)), np.zeros((nS, nA, nS))
    terminal = np.zeros(nS, dtype=bool)
    for s in range(nS):
        for a in range(nA):
            for p, s2, r, done in table[s][a]:
                P[s, a, s2] += p          # duplicates (e.g. two slips into the same wall) add up
                R[s, a, s2] = r
        terminal[s] = all(len(table[s][a]) == 1 and table[s][a][0][1] == s and table[s][a][0][3]
                          for a in range(nA))
    return P, R, terminal


def print_policy(env, pi, V=None):
    '''Policy as arrows on the map (H and G shown as letters), and the values as a grid.'''
    desc = env.unwrapped.desc.astype(str)
    arrows = np.array(list('←↓→↑'))[pi].reshape(desc.shape)
    grid = np.where(np.isin(desc, ['H', 'G']), desc, arrows)
    for row in grid:
        print(' '.join(row))
    if V is not None:
        print(V.reshape(desc.shape))


P, R, terminal = get_model(env)
print('P shape:', P.shape, '  terminal states:', np.flatnonzero(terminal))
print('rows of P sum to 1:', np.allclose(P.sum(axis=2), 1))
env.reset(seed=0)                     # Gymnasium requires reset() before render()
plt.figure(figsize=(4, 4)); plt.imshow(env.render()); plt.axis('off'); plt.title('FrozenLake 4x4'); plt.show()

### Watching an episode (provided)

`play(env, pi)` runs **one** episode with the policy `pi` and animates the environment's own `env.render()` frames, as in the search notebook.

`pi` is the deterministic policy $\pi\colon\mathcal S\to\mathcal A$ stored as an array, `pi[s]` = the action to take in state `s` (this is what value and policy iteration return); `pi=None` picks a random action every step.

**The elf always faces the direction it chose**. When it moves in a different direction, it slipped.

In [6]:
def play(env, pi=None, seed=None, title='', interval=400):
    '''Run one episode with policy pi (pi[s] = action; None = random) and animate env.render().'''
    s, _ = env.reset(seed=seed)
    env.action_space.seed(seed)
    frames, done = [env.render()], False
    while not done:
        a = env.action_space.sample() if pi is None else int(pi[s])
        s, r, terminated, truncated, _ = env.step(a)
        frames.append(env.render())
        done = terminated or truncated
    outcome = 'reached G' if r > 0 else ('fell into a hole' if terminated else 'time limit')

    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(frames[0])
    ax.axis('off')
    ax.set_title(f'{title}\n{outcome} after {len(frames) - 1} steps')

    def update(i):
        im.set_array(frames[i])
        return [im]

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode='once'))


play(env, pi=None, seed=3, title='random policy')

## Part 1. Q-values and the Bellman update (0.8 pt)

### Q1.1 (0.8 pt). Look at the model.
From state 14 (bottom row, just left of the goal), print the distribution of next states for each action. Which actions can reach the goal in one step, and with what probability? Why is it never 1, and which action can never reach it?

In [7]:
from fractions import Fraction as frac
for a in range(4):
    nxt = {int(s2): str(frac(p).limit_denominator().as_integer_ratio()[0])+'/'+str(frac(p).limit_denominator().as_integer_ratio()[1]) for s2, p in enumerate(P[14, a]) if p > 0}
    print(f'action {list('←↓→↑')[a]}: {nxt}')      # 0 = Left, 1 = Down, 2 = Right, 3 = Up

*Your answer here.*

### Q1.2 (0 pt). Q-values and the Bellman optimality update.

Implement
$$Q(s,a)=\sum_{s'}P(s'\mid s,a)\,\big[R(s,a,s')+\gamma V(s')\big],\qquad (BV)(s)=\max_a Q(s,a),$$
with $(BV)(s)=0$ for terminal states. Use array operations (no Python loops over states).

In [ ]:
def q_values(P, R, V, gamma):
    return np.einsum('ijk,ijk->ij', P, R + gamma * V[None, None, :]) # the summation over k after compoment-wise multiplication


def bellman_update(P, R, terminal, V, gamma):
    V_new = q_values(P, R, V, gamma).max(axis=1)
    V_new[terminal] = 0.0
    return V_new


# test: from V = 0, one update gives the probability of reaching G in ONE step under the best action
V1 = bellman_update(P, R, terminal, np.zeros(len(P)), gamma=0.99)
print(V1.reshape(4, 4))
assert np.isclose(V1[14], 1/3) and np.isclose(V1[0], 0)

## Part 2. Policy evaluation (1.5 pt)

Before searching for the best policy, we learn to compute the value of a **given** policy.

### Q2.1 (0 pt). Exact policy evaluation.
For a fixed deterministic policy $\pi$, $V^\pi$ solves the **linear** system
$$V^\pi = R^\pi + \gamma P^\pi V^\pi,\qquad P^\pi[s,s'] = P(s'\mid s,\pi(s)),\quad R^\pi[s]=\sum_{s'}P(s'\mid s,\pi(s))R(s,\pi(s),s'),$$
with $V^\pi(s)=0$ at terminal states. Implement it with `np.linalg.solve`. The provided code evaluates three policies: "always Left", "always Down" and a random one.

In [ ]:
def policy_evaluation(P, R, terminal, pi, gamma):
    n = len(P)
    P_pi = P[np.arange(n), pi]                                   # n x n
    R_pi = np.einsum('ij,ij->i', P_pi, R[np.arange(n), pi])
    A = np.eye(n) - gamma * P_pi
    A[terminal] = 0.0
    A[terminal, terminal] = 1.0                                  # equation V(s) = 0 for terminals
    R_pi[terminal] = 0.0
    return np.linalg.solve(A, R_pi)


always_left = np.zeros(len(P), dtype=int)
always_down = np.ones(len(P), dtype=int)
random_pi = np.random.default_rng(1).integers(4, size=len(P))     # a random deterministic policy
for name, pol in [('always Left', always_left), ('always Down', always_down), ('random', random_pi)]:
    V_pol = policy_evaluation(P, R, terminal, pol, gamma=0.99)
    print(f'{name}:  V(start) = {V_pol[0]:.4f}')
    print_policy(env, pol, V_pol)
    print()

### Q2.2 (1.5 pt). Policy evaluation by simple iterations.
The linear system $V=R^\pi+\gamma P^\pi V$ can also be solved by the fixed-point (simple) iteration
$$V_{k+1}=R^\pi+\gamma P^\pi V_k,\qquad V_0=0,\qquad V_{k+1}(s)=0 \text{ for terminal } s .$$

**Why it converges.** Write the update as $V_{k+1}=T^\pi V_k$ with $T^\pi V=R^\pi+\gamma P^\pi V$. For any two vectors $V,V'$,
$$\|T^\pi V-T^\pi V'\|_\infty=\gamma\,\|P^\pi(V-V')\|_\infty\le\gamma\,\|V-V'\|_\infty ,$$
because each row of $P^\pi$ is a probability distribution, so $(P^\pi u)(s)=\sum_{s'}P^\pi_{ss'}u(s')$ is an average of the entries of $u$ and cannot exceed $\max_{s'}|u(s')|$. A map that shrinks all distances by the factor $\gamma<1$ is a **contraction**.

**Consequences (Banach fixed-point theorem).**
* *At most one fixed point:* if $T^\pi V=V$ and $T^\pi V'=V'$, then $\|V-V'\|_\infty=\|T^\pi V-T^\pi V'\|_\infty\le\gamma\|V-V'\|_\infty$, which forces $V=V'$.
* *Convergence from any start, at a known rate:* the sequence $V_{k+1}=T^\pi V_k$ satisfies $\|V_{k+1}-V_k\|_\infty\le\gamma^k\|V_1-V_0\|_\infty$, so it is a Cauchy sequence and converges; its limit is a fixed point because $T^\pi$ is continuous, so it is $V^\pi$. Moreover
$$\|V_k-V^\pi\|_\infty=\|T^\pi V_{k-1}-T^\pi V^\pi\|_\infty\le\gamma\|V_{k-1}-V^\pi\|_\infty\le\dots\le\gamma^k\|V_0-V^\pi\|_\infty .$$
* *Stopping rule:* by the triangle inequality, $\|V_{k+1}-V^\pi\|_\infty\le\frac{\gamma}{1-\gamma}\|V_{k+1}-V_k\|_\infty$, so a small change between sweeps guarantees a small error.

The same argument, with one extra step for the max, will give the convergence of value iteration in Part 4.

Implement the iteration, stopping when $\|V_{k+1}-V_k\|_\infty<$ `tol`, and return $V$ and the number of sweeps.

Then (code below is provided): check that it agrees with the exact solution from Q2.1, and plot $\|V_k-V^\pi\|_\infty$ against $k$ for "always Down" with $\gamma=0.9$ and $\gamma=0.99$.

1. The bound $\gamma^k$ says that reaching the accuracy $10^{-12}$ may take up to $\log 10^{-12}/\log\gamma$ sweeps, about 260 for $\gamma=0.9$ and 2750 for $\gamma=0.99$. How many sweeps were actually needed? Why so few?
2. One sweep costs $O(|\mathcal S|^2)$, the linear solve $O(|\mathcal S|^3)$. When would you prefer the iterations?

In [ ]:
def policy_evaluation_iterative(P, R, terminal, pi, gamma, tol=1e-12, max_iter=100_000):
    n = len(P)
    P_pi = P[np.arange(n), pi]                                   # n x n
    R_pi = np.einsum('ij,ij->i', P_pi, R[np.arange(n), pi])      # expected one-step reward
    V = np.zeros(n)
    for k in range(1, max_iter + 1):
        V_new = R_pi + gamma * P_pi @ V
        V_new[terminal] = 0.0
        if np.abs(V_new - V).max() < tol:
            return V_new, k
        V = V_new
    return V, k

In [ ]:
for name, pol in [('always Down', always_down), ('random', random_pi)]:
    for gamma in [0.9, 0.99]:
        V_it, k = policy_evaluation_iterative(P, R, terminal, pol, gamma)
        V_ex = policy_evaluation(P, R, terminal, pol, gamma)
        print(f'{name:12s} γ = {gamma}:  sweeps = {k:4d}   max |V_iter - V_exact| = {np.abs(V_it - V_ex).max():.1e}')

# convergence of the simple iterations for "always Down"
n = len(P)
P_pi = P[np.arange(n), always_down]
R_pi = np.einsum('ij,ij->i', P_pi, R[np.arange(n), always_down])
for gamma, style in [(0.9, 'o-'), (0.99, 's-')]:
    V_ex = policy_evaluation(P, R, terminal, always_down, gamma)
    V, errs = np.zeros(n), []
    for k in range(100):
        errs.append(np.abs(V - V_ex).max())
        V = R_pi + gamma * P_pi @ V
        V[terminal] = 0.0
    ks = np.arange(100)
    line, = plt.semilogy(ks, errs, style, ms=2, label=f'‖V_k − V^π‖, γ = {gamma}')
    plt.semilogy(ks, gamma**ks * errs[0], '--', color=line.get_color(), label=f'γ^k bound, γ = {gamma}')
plt.xlabel('sweep k'); plt.ylim(1e-14, 1); plt.legend(); plt.grid(alpha=.3)
plt.title('Iterative evaluation of "always Down" on the slippery 4x4 lake'); plt.show()

*Your answers here.*

## Part 3. Policy iteration (0 pt)

Now improve the policy. Start from "always Left". Repeat: evaluate $\pi$ exactly (Q2.1), then improve it greedily, $\pi(s)\leftarrow\arg\max_a Q^\pi(s,a)$, using `q_values` from Q1.2. Stop when the policy no longer changes (change only where the improvement is larger than $10^{-12}$, to avoid flipping between tied actions).

By the policy improvement theorem each new policy is at least as good as the previous one, and there are only finitely many deterministic policies, so the loop stops, at an optimal policy. Report the number of iterations and print the final policy and values.

You can use the analitical solution for the value evaluation (small world).

In [ ]:
def policy_iteration(P, R, terminal, gamma, pi0=None):
    n = len(P)
    pi = np.zeros(n, dtype=int) if pi0 is None else pi0.copy()
    for k in range(1, 1000):
        V = policy_evaluation(P, R, terminal, pi, gamma)
        Q = q_values(P, R, V, gamma)
        best = Q.argmax(axis=1)
        better = Q[np.arange(n), best] > Q[np.arange(n), pi] + 1e-12
        if not better.any():
            return V, pi, k
        pi[better] = best[better]

V_pi, pi_pi, k_pi = policy_iteration(P, R, terminal, gamma=0.99)
print(f'policy iteration: {k_pi} iterations;  V*(start) = {V_pi[0]:.4f}')
print_policy(env, pi_pi, V_pi)

## Part 4. Value iteration (2 pt)

Policy iteration evaluates every policy completely. **Value iteration** skips that: it applies the Bellman optimality operator from Q1.2 again and again, $V_{k+1}=BV_k$, where
$$(BV)(s)=\max_a Q_V(s,a),\qquad Q_V(s,a)=\sum_{s'}P(s'\mid s,a)\,[R(s,a,s')+\gamma V(s')].$$
Each sweep is one step of evaluation with the improvement (the max) built in.

_In other words, it's a policy iteration with one iteration from the numerical (simple iterations) value evaluation._

### Q4.1 (1 pt). Implement value iteration.
Start from $V_0=0$ and repeat $V_{k+1}=BV_k$ until $\|V_{k+1}-V_k\|_\infty < \varepsilon(1-\gamma)/\gamma$. Return the final $V$, the greedy policy $\pi(s)=\arg\max_a Q(s,a)$, and the number of iterations. The second (provided) cell compares the result with policy iteration.

In [ ]:
def value_iteration(P, R, terminal, gamma, eps=1e-8, max_iter=100_000):
    V = np.zeros(len(P))
    tol = eps * (1 - gamma) / gamma
    for k in range(1, max_iter + 1):
        V_new = bellman_update(P, R, terminal, V, gamma)
        done = np.abs(V_new - V).max() < tol
        V = V_new
        if done:
            break
    pi = q_values(P, R, V, gamma).argmax(axis=1)
    return V, pi, k

V_star, pi_star, n_iter = value_iteration(P, R, terminal, gamma=0.99)
print(f'converged in {n_iter} iterations;  V*(start) = {V_star[0]:.4f}')
print_policy(env, pi_star, V_star)

In [ ]:
print(f'policy iteration: {k_pi} iterations;  value iteration: {n_iter} sweeps')
print('max |V_PI - V_VI| =', np.abs(V_pi - V_star).max())
print('max |V^pi* - V*|  =', np.abs(policy_evaluation(P, R, terminal, pi_star, 0.99) - V_star).max(),
      '  (exact evaluation of the greedy policy)')
_, k_eval = policy_evaluation_iterative(P, R, terminal, pi_star, gamma=0.99)
print(f'iterative evaluation of pi*: {k_eval} sweeps (compare with "always Down" in Q2.2)')

*Your comparison here: why does policy iteration need so few iterations, and what does each iteration cost? Why does evaluating $\pi^*$ by simple iterations take so many more sweeps than evaluating "always Down"?*

Watch the optimal agent. Try several seeds: you will see slips, and how the policy recovers from them.

In [ ]:
play(env, pi_star, seed=0, title='optimal policy, γ = 0.99')

### Q4.2 (1 pt). Deterministic vs. slippery ice.
Run value iteration on the **non-slippery** 4×4 lake (`make_env("4x4", is_slippery=False)`) with $\gamma=0.99$ and print both solutions with `print_policy`. Answer:

1. In the deterministic lake, what is $V^*(\text{start})$, and why does it equal $\gamma^{5}$?
2. In the slippery lake, look at the states next to holes. Why does the optimal policy often point **away** from where the agent wants to go (for example, Left in the start state)?

In [ ]:
env_det = make_env("4x4", is_slippery=False)
P_d, R_d, term_d = get_model(env_det)
V_d, pi_d, k_d = value_iteration(P_d, R_d, term_d, gamma=0.99)
print(f'deterministic: V*(start) = {V_d[0]:.4f},  0.99**5 = {0.99**5:.4f}')
print('deterministic:'); print_policy(env_det, pi_d, V_d)
print('slippery:');      print_policy(env, pi_star, V_star)

*Your answers here.*

## Part 5. The discount factor changes the policy (3 pt)

On the **8×8** slippery lake, compute the optimal policies for $\gamma=0.9$ and $\gamma=0.99$ and print them with `print_policy`.

1. Find at least two states where the two policies differ, and explain the difference in terms of "risky shortcut vs. safe detour".
2. $V^*(\text{start})$ is much smaller for $\gamma=0.9$. Does that mean the $\gamma=0.9$ policy reaches the goal less often? (You will measure this in Part 6.)

In [ ]:
env8 = make_env("8x8", is_slippery=True)
P8, R8, term8 = get_model(env8)

V8_09, pi8_09, _ = value_iteration(P8, R8, term8, gamma=0.9)
V8_99, pi8_99, _ = value_iteration(P8, R8, term8, gamma=0.99)
print(f'γ = 0.9,  V*(start) = {V8_09[0]:.3f}');  print_policy(env8, pi8_09)
print(f'γ = 0.99, V*(start) = {V8_99[0]:.3f}');  print_policy(env8, pi8_99)
differ = np.flatnonzero((pi8_09 != pi8_99) & ~term8)
print('states where the policies differ:', differ)

*Your answers here.*

## Part 6. Checking by simulation (1 pt)

Values are expected *discounted* returns; the game is won or lost. Write `success_rate(env, pi, n_episodes)` that plays `n_episodes` episodes with the deterministic policy `pi` and returns the fraction that reach the goal. (An episode ends when `terminated` or `truncated` is true; the time limit is 100 steps on 4×4 and 200 on 8×8.)

Then fill in the table: 4×4 with $\pi^*$ at $\gamma=0.99$, "always Left", a uniformly random policy (choose a random action each step), and 8×8 with the $\gamma=0.9$ and $\gamma=0.99$ policies. Use at least 2000 episodes each.

**Question.** On 4×4 the best policy wins only about three games in four. Is that a failure of value iteration? Explain.

In [ ]:
def success_rate(env, pi, n_episodes=2000, seed=0):
    '''pi: array of actions per state, or None for a uniformly random policy.'''
    rng_ = np.random.default_rng(seed)
    wins = 0
    for ep in range(n_episodes):
        s, _ = env.reset(seed=int(rng_.integers(1 << 31)))
        done = False
        while not done:
            a = int(rng_.integers(4)) if pi is None else int(pi[s])
            s, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
        wins += r > 0
    return wins / n_episodes


results = {
    '4x4, optimal (γ=0.99)': success_rate(env, pi_star),
    '4x4, always Left':      success_rate(env, always_left),
    '4x4, random':           success_rate(env, None),
    '8x8, optimal γ=0.9':    success_rate(env8, pi8_09),
    '8x8, optimal γ=0.99':   success_rate(env8, pi8_99),
}
for k, v in results.items():
    print(f'{k:24s} {v:.3f}')

Watch one 8×8 episode for each discount factor (change the seed to see others). Where does the γ = 0.9 agent take a risk that the γ = 0.99 agent avoids?

In [ ]:
display(play(env8, pi8_09, seed=1, title='8x8, γ = 0.9'))
display(play(env8, pi8_99, seed=1, title='8x8, γ = 0.99'))

*Your answer here.*

## Part 7. The contraction bound in practice (1 pt)

The theory is in Parts 2 and 4. Here you check the bound for **value iteration**.

On the slippery 4×4 lake with $\gamma=0.9$, plot $\|V_k-V^*\|_\infty$ against $k$ on a log scale together with the bound $\gamma^k\|V_0-V^*\|_\infty$ (use a very accurate $V^*$ as the reference). Is the bound tight here? Why does the error fall faster than the bound?

In [ ]:
gamma = 0.9
V_true, _, _ = value_iteration(P, R, terminal, gamma, eps=1e-14)
V, errs = np.zeros(len(P)), []
for k in range(60):
    errs.append(np.abs(V - V_true).max())
    V = bellman_update(P, R, terminal, V, gamma)
ks = np.arange(60)
plt.semilogy(ks, errs, 'o-', ms=3, label='‖V_k − V*‖')
plt.semilogy(ks, gamma**ks * errs[0], '--', label='γ^k ‖V_0 − V*‖')
plt.xlabel('k'); plt.legend(); plt.grid(alpha=.3); plt.show()

*Your answer here.*

## Bonus (2.5 pt): your own lake

Design a map, either by hand, e.g. `gym.make("FrozenLake-v1", desc=["SFFH", "FHFF", "FFFH", "HFFG"], is_slippery=True)`, or with `gymnasium.envs.toy_text.frozen_lake.generate_random_map(size=8, p=0.8)`, on which the optimal **slippery** policy and the optimal **deterministic** policy disagree in at least half of the non-terminal states. Show both policies and explain one disagreement.